## The daemon configuration — `daemon.json`

`dockerd` reads **`/etc/docker/daemon.json`** at startup. It's optional — no file means defaults — but it's the one place you tune the engine. The schema is huge; these are the keys that matter in practice:

```json
{
  "data-root": "/mnt/docker-data",
  "storage-driver": "overlay2",
  "log-driver": "json-file",
  "log-opts": { "max-size": "10m", "max-file": "3" },
  "default-runtime": "runc",
  "registry-mirrors": ["https://my-cache.local"],
  "insecure-registries": ["registry.lab.internal:5000"],
  "userns-remap": "default",
  "userland-proxy": false,
  "live-restore": true
}
```

The ones you'll actually reach for, grouped by the module they connect to:

- **`data-root`** — move Docker's data dir off the root partition onto a dedicated disk (the fix for "`/` is full").
- **`storage-driver`** / **`log-driver`** + **`log-opts`** — the drivers from the next section; `log-opts` with `max-size`/`max-file` is the **log-rotation** every prod host needs (module 03's unbounded-`json-file` warning).
- **`registry-mirrors`** / **`insecure-registries`** — point at the pull-through cache (module 07) and allow a plain-HTTP internal registry.
- **`userns-remap`** / **`userland-proxy`** — the security and networking toggles from modules 08 and 05.
- **`live-restore: true`** — a big one: **keep containers running while the daemon is restarted or upgraded**, so a `dockerd` update doesn't take your workloads down.

Two operational habits: after editing, **validate before restarting** — `sudo dockerd --validate` catches a syntax error or unknown key *before* it stops the daemon coming back up — then `sudo systemctl restart docker`. A malformed `daemon.json` is the single most common reason a daemon won't start, which is exactly where troubleshooting begins.